# 08-classifier — Bộ phân lớp Gaussian và bảng S0

**Câu hỏi**: argmax của bộ phân lớp do số hạng nào của log-mật độ quyết định (độ lệch −½ log det Σ, số hạng bậc hai, số hạng tuyến tính), cách đó khác công thức của paper (Eq. 4, 6) thế nào, và lỗi trên test S0 nằm ở đâu? (D24, Q08.01, Q08.03)

### Nguồn gốc (provenance)

In commit và trạng thái dirty của code, rồi đặt `XLA_FLAGS` trước mọi lần import jax (mẫu ở `lab/README.vi.md` §11).

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit 59b9a56 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `59b9a56`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 08

Dùng `replay.rerun`: nạp ảnh chụp sau bước 07 của `gc10_v2_ft_r0_s4`, chạy lại `s08_classifier`, rồi so hai khoá với `snap/08.pkl`.
- `predict`: tên hàm phân lớp mà 12 và 14 dùng lại.
- `acc0`: bảng S0 với bốn dòng All, Old, Novel, S0.

Quy tắc đạt như mọi bước (QD21/C1): khoá rời rạc trùng tuyệt đối; số thực lệch ≤ 1e-6 × max|giá trị|.

Dự đoán: 2/2 khoá đạt. Cả bốn dòng của `acc0` bằng 85.477% (206/241, S0_All của run), vì ở S0 cả bốn dòng đều chấm trên `test0` (`04-loader` §3.1).

In [2]:
# cell 1: rerun step 08 of r0 s4 from the snapshot of step 07 and compare with snap/08.pkl
RUN = "gc10_v2_ft_r0_s4"
out_11, ok_11 = replay.rerun(RUN, "08")
print("predict:", out_11["predict"])
print("acc0:", {k: round(v, 3) for k, v in out_11["acc0"].items()})

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 08 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
+---------+---------+---------+-------------------+
| TestSet | Correct | Smaples |      Accuracy     |
+---------+---------+---------+-------------------+
|   All   |   206   |   241   | 85.47717842323651 |
|   Old   |   206   |   241   | 85.47717842323651 |
|  Novel  |   206   |   241   | 85.47717842323651 |
|    S0   |   206   |   241   | 85.47717842323651 |
+---------+---------+---------+-------------------+
  OK   predict      str                    max|diff| nan
  OK   acc0         dict                   max|diff| 0
rerun step 08: OK (2/2 keys)
predict: predict_default
acc0: {'All': 85.477, 'Old': 85.477, 'Novel': 85.477, 'S0': 85.477}


- **Kết quả**:
  - `rerun` bước 08 đạt 2/2 khoá: `predict` = `predict_default`; `acc0` trùng ảnh chụp (max|diff| = 0), dù driver GPU đã đổi từ 617.42 sang 621.01.
  - Bảng S0: cả bốn dòng All, Old, Novel, S0 đều 206/241 = 85.477%.
  - Cảnh báo `TqdmWarning` (thiếu ipywidgets) chỉ là thanh tiến trình, không ảnh hưởng kết quả.
- **Ý nghĩa**:
  - Đúng như dự đoán: bước 08 tái lập được, và ở S0 bốn dòng của bảng là một con số, vì cả bốn chấm trên `test0`.
  - Bước 08 không học gì. Nó chỉ chọn cách phân lớp (`predict_default`) và chấm `params0` của 06. Phần còn lại của notebook mổ chính cách phân lớp đó.

## 2. Code

### 2.1 Bộ phân lớp trong code

In nguyên văn bằng `inspect.getsource`:
- `replay.s08_classifier`: đặt tham số là `params0`, `label_offset` 0, rồi chấm bảng S0;
- `replay.predict_default`: gỡ mọi phép thay hàm `_predict` mà một biến thể để lại;
- `replay._table`: gọi `clf.test` cho từng dòng của bảng;
- `MNGMMClassifier.test`: chiếu qua PCA rồi gọi `_predict`;
- `MNGMMClassifier._predict`: log-mật độ của từng khe (slot), rồi argmax;
- `numpyro.distributions.MultivariateNormal.log_prob`: công thức log-mật độ thật sự được dùng.

Cần xem ba điều:
- argmax lấy trên bao nhiêu khe (D24);
- log-mật độ có số hạng log det hay không;
- `_predict` có trọng số tiên nghiệm (prior) của lớp hay không.

In [3]:
# cell 2: the code of step 08 and of the log-density it calls, printed verbatim; numpyro wraps log_prob in validate_sample
#         without __wrapped__, so the original function is taken from the wrapper's closure (log_prob_fn)
import inspect
import numpyro.distributions as dist_21
from classifier.mngmm import MNGMMClassifier
wrap_21 = dist_21.MultivariateNormal.log_prob
inner_21 = dict(zip(wrap_21.__code__.co_freevars, (c.cell_contents for c in wrap_21.__closure__)))["log_prob_fn"]
for f_21 in (replay.s08_classifier, replay.predict_default, replay._table, MNGMMClassifier.test,
             MNGMMClassifier._predict, inner_21):
    print(f"# ---- {f_21.__qualname__}  ({Path(inspect.getsourcefile(f_21)).name})")
    print(inspect.getsource(f_21))
import numpy as np
mvn_21 = dist_21.MultivariateNormal(np.zeros(2), np.eye(2))
print("validate_args of a MultivariateNormal built as in _predict:", mvn_21._validate_args)

# ---- s08_classifier  (replay.py)
def s08_classifier(st):
    """08 Classifier (used by 12 and 14) and the S0 test table, printed as mngmm.run prints it.
    In: params0, test0_x, true_test0_y. Out: predict, acc0."""
    clf = _clf(st)
    predict_default(clf)
    clf.params = clf.global_params = _jnp(st["params0"])
    clf.label_offset, clf.n_seen = 0, clf.num_base
    rows = [(n, st["test0_x"], st["true_test0_y"]) for n in ("All", "Old", "Novel", "S0")]
    return dict(predict=predict_default.__name__, acc0=_table(clf, rows))

# ---- predict_default  (replay.py)
def predict_default(clf):
    """Classifier: argmax of the class log-densities over all num_classes slots (mngmm._predict)."""
    clf.__dict__.pop("_predict", None)

# ---- _table  (replay.py)
def _table(clf, rows):
    """The result table of mngmm.run(), same format (scripts/extract_vbcgcd.py reads it)."""
    from prettytable import PrettyTable
    t, acc = PrettyTable(["TestSet", "Correct", "Smaples", "Accuracy"]), {}
  

- **Kết quả**:
  - `s08_classifier` đặt `params` = `params0`, `label_offset` = 0, rồi chấm bốn dòng trên `test0`.
  - `predict_default` chỉ gỡ phép thay hàm `_predict`. `_table` gọi `clf.test` cho từng dòng. `test` chiếu qua PCA (`pre_processing`) rồi gọi `_predict`.
  - `_predict`: với mỗi khe i < `class_means.shape[0]` (10 khe), tính `MultivariateNormal(μ_i, Σ_i).log_prob(X)`, rồi argmax. Chỉ cắt khe chưa gặp khi `mask_unseen` bật (mặc định tắt). Không có trọng số tiên nghiệm (prior) của lớp.
  - `MultivariateNormal.log_prob` = −½ M − (½ log det Σ + (d/2) log 2π). Trong đó M = (x − μ)ᵀΣ⁻¹(x − μ), tính qua thừa số Cholesky `scale_tril` của Σ. Hàm bọc `validate_sample` không đổi kết quả, vì `_validate_args` là False.
- **Ý nghĩa**:
  - Bộ phân lớp là biệt thức bậc hai (quadratic discriminant analysis) với tiên nghiệm đều: argmax_i [−½ log det Σ_i − ½ (x − μ_i)ᵀΣ_i⁻¹(x − μ_i)] (Hastie, Tibshirani & Friedman, ESL §4.3).
  - Code có số hạng log det. Eq. 4 và 6 của paper (NCM với khoảng cách Mahalanobis) thì không có. 3.2 đo xem số hạng này đổi bao nhiêu dự đoán.
  - Argmax lấy trên cả 10 khe, kể cả 5 khe chưa học N(0, I) ở S0 (D24).

## 3. Bộ phân lớp quyết định bằng gì

### 3.1 Tách đúng log-mật độ thành các số hạng

Với khe c (μ_c, Σ_c) và ảnh x (64 chiều PCA), log-mật độ tách đúng, không xấp xỉ, thành:

log N(x | μ_c, Σ_c) = b_c + q_c(x) + ℓ_c(x) + k_c + chung(x)

với:
- độ lệch theo lớp (bias): b_c = −½ log det Σ_c, không phụ thuộc x;
- số hạng bậc hai (quadratic): q_c(x) = −½ xᵀ(Σ_c⁻¹ − I)x;
- số hạng tuyến tính (linear): ℓ_c(x) = μ_cᵀΣ_c⁻¹x;
- hằng: k_c = −½ μ_cᵀΣ_c⁻¹μ_c;
- chung(x) = −½‖x‖² − (d/2) log 2π, giống nhau ở mọi khe nên không đổi argmax.

Suy dẫn: khai triển (x − μ)ᵀΣ⁻¹(x − μ) = xᵀΣ⁻¹x − 2μᵀΣ⁻¹x + μᵀΣ⁻¹μ (Σ đối xứng), rồi viết xᵀΣ⁻¹x = xᵀ(Σ⁻¹ − I)x + ‖x‖².

Hệ quả: với khe chưa học (μ = 0, Σ = I), cả bốn số hạng đều bằng 0. Vì vậy b_c + q_c + ℓ_c + k_c chính là điểm của khe c so với một khe N(0, I).

**Cách làm**:
- Tính bốn số hạng bằng numpy float64 trên `z_test0` (241 ảnh) với `params0`.
- Kiểm hai điều:
  - tổng các số hạng bằng log-mật độ float32 mà `_predict` trả về (sai lệch cỡ sai số float32);
  - argmax cho lại 206/241.
- In "độ trải" của từng số hạng giữa 5 lớp cũ: trung bình qua các ảnh của (max_c − min_c). Số hạng nào trải rộng nhất thì có sức đổi argmax lớn nhất.
- In bốn số hạng của ảnh test đầu tiên để tính tay.

Dự đoán [SUY LUẬN, từ `06-gauss-svi` §3.1–3.2 và `05-pca` §3.2]:
- b_c trải 0.29 (từ +0.42 ở welding_line tới +0.71 ở punching_hole);
- ℓ_c trải dưới 1, vì ‖μ‖ ≈ 0.09 và ‖x‖ ≈ 13–16;
- q_c trải vài đơn vị, vì Σ có trị riêng tới 1.32 và ‖x‖² ≈ 150–250;
- như vậy q_c, tức độ lệch của Σ so với I, quyết định nhiều nhất.

In [4]:
# cell 3: exact split of the log-density of each slot: log N(x|mu,S) = b + q + l + k + common, with b = -1/2 log det S,
#         q = -1/2 x'(S^-1 - I)x, l = mu'S^-1 x, k = -1/2 mu'S^-1 mu, common = -1/2|x|^2 - d/2 log 2pi (the same for every slot);
#         float64 numpy, checked against the float32 log-densities of _predict; the first test image term by term
import pandas as pd
import jax.numpy as jnp
st_31 = replay.load(RUN, "08")
mu_31 = np.asarray(st_31["params0"]["class_means"], np.float64)       # (10, 64)
S_31 = np.asarray(st_31["params0"]["class_covs"], np.float64)         # (10, 64, 64)
X_31 = np.asarray(st_31["z_test0"], np.float64)                        # (241, 64)
y_31 = np.asarray(st_31["true_test0_y"]).astype(int)
d_31 = X_31.shape[1]
P_31 = np.linalg.inv(S_31)
b_31 = -0.5 * np.linalg.slogdet(S_31)[1]                               # (10,)
q_31 = -0.5 * np.einsum("ni,cij,nj->nc", X_31, P_31 - np.eye(d_31), X_31)   # (241, 10)
l_31 = np.einsum("ci,cij,nj->nc", mu_31, P_31, X_31)                   # (241, 10)
k_31 = -0.5 * np.einsum("ci,cij,cj->c", mu_31, P_31, mu_31)            # (10,)
common_31 = -0.5 * (X_31 ** 2).sum(1) - 0.5 * d_31 * np.log(2 * np.pi)
logp_31 = b_31 + q_31 + l_31 + k_31 + common_31[:, None]

clf_31 = st_31["clf"]
pred_jax_31, lp_jax_31 = clf_31._predict(jnp.asarray(st_31["z_test0"]), replay._jnp(st_31["params0"]))
lp_jax_31 = np.asarray(lp_jax_31, np.float64)
print(f"sum of terms vs _predict (float32): max|diff| {np.abs(logp_31 - lp_jax_31).max():.3g}, "
      f"max|log-density| {np.abs(lp_jax_31).max():.4g}")
print("argmax equal on", int((logp_31.argmax(1) == np.asarray(pred_jax_31)).sum()), "of", len(y_31),
      "| correct: float64", int((logp_31.argmax(1) == y_31).sum()), ", JAX", int((np.asarray(pred_jax_31) == y_31).sum()))
print("slots 5-9 (untouched): max|b|, |q|, |l|, |k| =", [float(np.abs(t).max()) for t in (b_31[5:], q_31[:, 5:], l_31[:, 5:], k_31[5:])])
print("b (classes 0-4):", b_31[:5].round(3).tolist(), "| k:", k_31[:5].round(4).tolist())
spread_31 = lambda T: float((T[:, :5].max(1) - T[:, :5].min(1)).mean())
print("mean spread over the 5 old classes (max - min per image): b {:.3f}, q {:.3f}, l {:.3f}, k {:.4f}".format(
      float(b_31[:5].max() - b_31[:5].min()), spread_31(q_31), spread_31(l_31), float(k_31[:5].max() - k_31[:5].min())))
print("|x|^2 of the test images: median", round(float(np.median((X_31 ** 2).sum(1))), 1))
i_31 = 0
print(f"\ntest image {i_31}: true class {y_31[i_31]}, |x|^2 = {(X_31[i_31] ** 2).sum():.2f}, common = {common_31[i_31]:.3f}")
print(pd.DataFrame({"b": b_31[:5], "q": q_31[i_31, :5], "l": l_31[i_31, :5], "k": k_31[:5],
                    "b+q+l+k": (b_31 + q_31[i_31] + l_31[i_31] + k_31)[:5], "log-density": logp_31[i_31, :5]},
                   index=[f"class {c}" for c in range(5)]).round(4).to_string())

sum of terms vs _predict (float32): max|diff| 3.62e-05, max|log-density| 190.6
argmax equal on 241 of 241 | correct: float64 206 , JAX 206
slots 5-9 (untouched): max|b|, |q|, |l|, |k| = [0.0, 0.0, 0.0, 0.0]
b (classes 0-4): [0.708, 0.419, 0.567, 0.433, 0.415] | k: [-0.0032, -0.0037, -0.0024, -0.0037, -0.0044]
mean spread over the 5 old classes (max - min per image): b 0.294, q 10.998, l 0.746, k 0.0020
|x|^2 of the test images: median 191.1

test image 0: true class 0, |x|^2 = 200.24, common = -158.932
              b       q       l       k  b+q+l+k  log-density
class 0  0.7083  3.7883  0.2476 -0.0032   4.7410    -154.1910
class 1  0.4191  5.7255  0.3451 -0.0037   6.4860    -152.4460
class 2  0.5673 -0.3317 -0.0451 -0.0024   0.1881    -158.7439
class 3  0.4332 -1.6007 -0.1925 -0.0037  -1.3638    -160.2957
class 4  0.4145  0.2230 -0.2879 -0.0044   0.3453    -158.5867


- **Kết quả**:
  - Tổng các số hạng (float64) khớp log-mật độ float32 của `_predict` tới 3.6e-05 (log-mật độ lớn nhất 190.6). Argmax trùng ở 241/241 ảnh; cả hai cách đều cho 206 ảnh đúng.
  - Khe 5–9 (chưa học): b, q, ℓ, k đều bằng đúng 0.
  - b của lớp 0–4: 0.708, 0.419, 0.567, 0.433, 0.415. k: −0.0024 tới −0.0044.
  - Độ trải trung bình giữa 5 lớp cũ (max − min trên mỗi ảnh): q 10.998, ℓ 0.746, b 0.294, k 0.0020. Trung vị của ‖x‖² trên test là 191.1.
  - Ảnh test 0 (lớp thật 0, punching_hole; ‖x‖² = 200.24; phần chung −158.932):
    - lớp 0: b + q + ℓ + k = 0.7083 + 3.7883 + 0.2476 − 0.0032 = 4.7410, cộng phần chung thành −154.191 (tự cộng, khớp cột log-mật độ);
    - lớp 1 (welding_line) có q = 5.7255, cao nhất, nên tổng 6.4860 cao nhất. Ảnh bị đoán là welding_line.
    - b nghiêng về lớp 0 (+0.29) và ℓ nghiêng về lớp 1 (+0.10), nhưng cả hai nhỏ hơn hiệu của q (1.94).
- **Ý nghĩa**:
  - Phép tách đúng và đủ: tổng bốn số hạng cộng phần chung cho lại đúng log-mật độ và đúng quyết định của pipeline.
  - Dự đoán đúng, và chênh lệch lớn hơn dự đoán: q trải 11.0, gấp khoảng 15 lần ℓ (0.75) và 37 lần b (0.29) (tự chia).
  - Như vậy bộ phân lớp quyết định chủ yếu bằng độ lệch của Σ khỏi I, qua dạng toàn phương −½ xᵀ(Σ⁻¹ − I)x. Giải thích ở `06-gauss-svi` §1.1 ("Σ ≈ I nên argmax là μ_cᵀx") sai: phần tử của Σ − I nhỏ, nhưng nhân với ‖x‖² ≈ 190 thì thành lớn.
  - Ảnh 0 là một ví dụ của Q08.03: một ảnh punching_hole bị đoán là welding_line, đúng cặp lớp quanh mối hàn. 4.1 xem ảnh này có hộp welding_line không.
  - 3.2 đo xem trên cả 241 ảnh, số hạng nào một mình đã đủ cho phần lớn quyết định.

### 3.2 Bộ phân lớp chỉ dùng một phần số hạng; bản đọc theo paper

Hỏi: nếu chỉ giữ một vài số hạng thì bộ phân lớp còn đúng bao nhiêu và trùng quyết định với pipeline bao nhiêu? Đặc biệt, bỏ b_c (Eq. 4 và 6 của paper không có log det) thì đổi bao nhiêu ảnh?

**Cách làm**: argmax trên đủ 10 khe, như `_predict`, của các tổng sau (khe chưa học luôn được 0):
- `full` = b + q + ℓ + k: pipeline;
- `no_b` = q + ℓ + k: đọc theo paper, khoảng cách Mahalanobis không có log det;
- `q`, `l`, `b`: từng số hạng một;
- `q+b`: bỏ phần của μ;
- `l+b+k`: bỏ phần của Σ − I trong dạng toàn phương.

Với mỗi bộ, in số ảnh đúng trên 241, số ảnh cùng nhãn với `full`, và số ảnh rơi vào khe 5–9.

Thêm phân rã khoảng cách quyết định của `full`: với mỗi ảnh, lấy lớp thắng và lớp đứng thứ hai; hiệu của từng số hạng giữa hai lớp đó (Δb, Δq, Δℓ) cộng lại thành khoảng cách. In trung bình |Δ| của từng số hạng, và tỉ lệ ảnh mà |Δq| lớn nhất trong ba hiệu.

Dự đoán [SUY LUẬN, từ 3.1]: `q` một mình gần bằng `full`; `l` thấp hẳn; `no_b` chỉ đổi vài ảnh, vì b trải 0.29 so với q 11.

In [5]:
# cell 4: classifiers that keep only some terms (argmax over the 10 slots, as _predict; untouched slots score 0), against the
#         pipeline (full); the paper's reading drops b (Eq. 4, 6: no log det); split of the winner-vs-runner-up margin of full
B_32 = np.broadcast_to(b_31, q_31.shape)
K_32 = np.broadcast_to(k_31, q_31.shape)
variants_32 = {"full": B_32 + q_31 + l_31 + K_32, "no_b (paper)": q_31 + l_31 + K_32, "q": q_31, "l": l_31, "b": B_32,
               "q+b": q_31 + B_32, "l+b+k": l_31 + B_32 + K_32}
full_32 = variants_32["full"].argmax(1)
rows_32 = []
for name_32, T in variants_32.items():
    p = T.argmax(1)
    rows_32.append(dict(classifier=name_32, correct=int((p == y_31).sum()), same_as_full=int((p == full_32).sum()),
                        to_slots_5_9=int((p >= 5).sum())))
print(pd.DataFrame(rows_32).to_string(index=False))

top2_32 = np.argsort(-variants_32["full"], axis=1)[:, :2]                # winner, runner-up per image
w_32, r_32 = top2_32[:, 0], top2_32[:, 1]
idx_32 = np.arange(len(y_31))
dB_32, dQ_32, dL_32 = (T[idx_32, w_32] - T[idx_32, r_32] for T in (B_32, q_31, l_31))
print("\nmargin winner - runner-up of full: mean |db| {:.3f}, |dq| {:.3f}, |dl| {:.3f}".format(
      np.abs(dB_32).mean(), np.abs(dQ_32).mean(), np.abs(dL_32).mean()))
big_32 = np.argmax(np.abs(np.stack([dB_32, dQ_32, dL_32], 1)), 1)
print("largest |delta| is b / q / l on", np.bincount(big_32, minlength=3).tolist(), "of", len(y_31), "images")
print("delta q alone points to the winner (dq > 0) on", int((dQ_32 > 0).sum()), "images;",
      "b alone on", int((dB_32 > 0).sum()), "; l alone on", int((dL_32 > 0).sum()))

  classifier  correct  same_as_full  to_slots_5_9
        full      206           241             0
no_b (paper)      206           241             0
           q      205           240             0
           l      200           226             0
           b       43            49             0
         q+b      204           239             0
       l+b+k      189           207             0

margin winner - runner-up of full: mean |db| 0.241, |dq| 6.795, |dl| 0.407
largest |delta| is b / q / l on [3, 237, 1] of 241 images
delta q alone points to the winner (dq > 0) on 240 images; b alone on 155 ; l alone on 228


- **Kết quả** (số ảnh trên 241):

  | bộ phân lớp | đúng | cùng nhãn với `full` | vào khe 5–9 |
  |---|---|---|---|
  | `full` (pipeline) | 206 | 241 | 0 |
  | `no_b` (đọc theo paper) | 206 | 241 | 0 |
  | `q` | 205 | 240 | 0 |
  | `l` | 200 | 226 | 0 |
  | `b` | 43 | 49 | 0 |
  | `q+b` | 204 | 239 | 0 |
  | `l+b+k` | 189 | 207 | 0 |

  - Khoảng cách giữa lớp thắng và lớp thứ hai của `full`: trung bình |Δb| 0.241, |Δq| 6.795, |Δℓ| 0.407.
  - |Δq| lớn nhất trong ba hiệu ở 237/241 ảnh (b: 3, ℓ: 1).
  - Một mình Δq chỉ về lớp thắng ở 240/241 ảnh, Δℓ ở 228, Δb ở 155.
- **Ý nghĩa**:
  - Quyết định của pipeline gần như hoàn toàn là quyết định của q. Bộ chỉ dùng q trùng nhãn với pipeline ở 240/241 ảnh.
  - Bỏ b (đọc theo paper) không đổi ảnh nào ở S0. Khác biệt paper–code về log det (ứng viên D25) vô hại ở S0. Nó chỉ có thể có tác dụng ở S1, nơi b của khe mới khác b của khe cũ; đo ở 12 và 14.
  - `b` một mình cho 43 ảnh đúng: mọi ảnh đều về punching_hole, lớp có b lớn nhất, và test có 43 ảnh punching_hole (tự đối chiếu với file split).
  - Dự đoán đúng một nửa. q quyết định, đúng. Nhưng ℓ không "thấp hẳn": μ một mình vẫn đúng 200/241. Như vậy μ mang đủ hướng của lớp; chỉ vì quá nhỏ (‖μ‖ ≈ 0.09) nên nó bị q lấn. Khi q và ℓ chỉ khác lớp, q thắng.
  - D24 ở r0 s4 với SVI: không ảnh nào vào khe 5–9 với bất kỳ bộ nào, vì các khe đã học được q dương với ảnh của lớp mình. 5.1 kiểm trên 15 ảnh chụp.
  - [SUY LUẬN] Hệ quả cho S1: sức hút của một khe chủ yếu là độ lớn q của nó, mà q lớn dần theo số bước Adam. Khe mới ở lượt S chỉ học 468–620 bước, khe cũ học 1000 bước. Vì vậy ở 14 có thể có thiên lệch về khe cũ. Kiểm ở 12 và 14 bằng chính phép tách này.

### 3.3 q lấy thông tin từ đâu: hướng riêng chính của Σ_c − I

Hỏi ba điều:
1. Hướng riêng u có |λ − 1| lớn nhất của Σ_c nằm theo tâm lớp x̄_c, theo các thành phần mang lớp (pc1–pc4), hay theo các thành phần mang camera (pc5, pc8, pc12, pc16; `05a-camera` §2.1)?
2. Adam đẩy Σ_c về phía nào? Gradient của log-likelihood theo Σ tại μ ≈ 0, Σ ≈ I tỉ lệ với M_c − I, với M_c = (1/n) Σ x xᵀ là moment bậc hai quanh gốc (Petersen & Pedersen, *The Matrix Cookbook* §2.1–2.2). Nếu μ đã tới tâm thì gradient lại tỉ lệ với S_c − I, hiệp phương sai quanh tâm. Vì μ gần như chưa rời gốc (`06-gauss-svi` §3.1), dự đoán là phía M_c.
3. Chỉ một hướng u có đủ cho phân lớp không?

**Cách làm**, với mỗi lớp cũ c, trên `z_s0` (764 ảnh S0):
- λ, u của Σ_c; cos(u, x̄_c) (lấy trị tuyệt đối, vì u chỉ xác định tới dấu);
- tỉ lệ năng lượng của u (Σ u_j²) nằm ở pc1–pc4, và ở pc5, pc8, pc12, pc16. Mốc ngẫu nhiên là 4/64 = 0.0625 cho mỗi nhóm;
- trên các phần tử ngoài đường chéo (95% năng lượng của E, `06-gauss-svi` §3.2): tỉ lệ phần tử mà dấu của E_c trùng dấu của M_c, và trùng dấu của S_c;
- bộ phân lớp hạng 1: q chỉ giữ hướng u, tức −½(1/λ − 1)(uᵀx)², argmax trên 10 khe.

Dự đoán [SUY LUẬN]: |cos(u, x̄_c)| cao; năng lượng của u dồn ở pc1–pc4; dấu của E khớp M nhiều hơn S; bộ hạng 1 gần bằng `q`.

In [6]:
# cell 5: where q comes from: the eigen-direction u of each old-class Sigma with the largest |lambda - 1|, against the class
#         centre and the class / camera components (05a-camera 2.1); signs of the off-diagonal E = Sigma - I against the second
#         moment about the origin M and the covariance S of the class; a rank-1 q classifier
Z0_33 = np.asarray(st_31["z_s0"], np.float64)
y0_33 = np.asarray(st_31["s0_y"]).astype(int)
cam_33 = [4, 7, 11, 15]                                                   # pc5, pc8, pc12, pc16
off_33 = ~np.eye(d_31, dtype=bool)
q1_33 = np.zeros_like(q_31)                                               # untouched slots stay 0
rows_33 = []
for c in range(5):
    lam, U = np.linalg.eigh(S_31[c])
    j = int(np.argmax(np.abs(lam - 1)))
    u = U[:, j]
    Xc = Z0_33[y0_33 == c]
    xbar = Xc.mean(0)
    M = Xc.T @ Xc / len(Xc)                                               # second moment about the origin
    Sc = np.cov(Xc.T, bias=True)                                          # covariance about the class centre
    E = S_31[c] - np.eye(d_31)
    rows_33.append(dict(cls=c, lam_u=round(float(lam[j]), 3), lam_range=f"{lam.min():.3f}-{lam.max():.3f}",
                        cos_u_xbar=round(abs(float(u @ xbar)) / float(np.linalg.norm(xbar)), 3),
                        energy_pc1_4=round(float((u[:4] ** 2).sum()), 3), energy_cam=round(float((u[cam_33] ** 2).sum()), 3),
                        sign_E_eq_M=round(float((np.sign(E[off_33]) == np.sign(M[off_33])).mean()), 3),
                        sign_E_eq_S=round(float((np.sign(E[off_33]) == np.sign(Sc[off_33])).mean()), 3)))
    q1_33[:, c] = -0.5 * (1 / lam[j] - 1) * (X_31 @ u) ** 2
print(pd.DataFrame(rows_33).to_string(index=False))
p1_33 = q1_33.argmax(1)
print("\nrank-1 q classifier: correct", int((p1_33 == y_31).sum()), "of", len(y_31), "| same as full", int((p1_33 == full_32).sum()))

 cls  lam_u   lam_range  cos_u_xbar  energy_pc1_4  energy_cam  sign_E_eq_M  sign_E_eq_S
   0  1.318 0.824-1.318       0.068         0.103       0.145        0.933        0.927
   1  1.267 0.832-1.267       0.035         0.051       0.145        0.958        0.942
   2  1.324 0.831-1.324       0.080         0.159       0.094        0.942        0.935
   3  1.271 0.827-1.271       0.091         0.108       0.279        0.958        0.949
   4  1.303 0.844-1.303       0.029         0.008       0.044        0.946        0.932

rank-1 q classifier: correct 91 of 241 | same as full 98


- **Kết quả**:

  | lớp | λ của u | khoảng λ | \|cos(u, x̄)\| | năng lượng u ở pc1–pc4 | ở pc5, 8, 12, 16 | dấu E trùng M | dấu E trùng S |
  |---|---|---|---|---|---|---|---|
  | 0 punching_hole | 1.318 | 0.824–1.318 | 0.068 | 0.103 | 0.145 | 0.933 | 0.927 |
  | 1 welding_line | 1.267 | 0.832–1.267 | 0.035 | 0.051 | 0.145 | 0.958 | 0.942 |
  | 2 crescent_gap | 1.324 | 0.831–1.324 | 0.080 | 0.159 | 0.094 | 0.942 | 0.935 |
  | 3 water_spot | 1.271 | 0.827–1.271 | 0.091 | 0.108 | 0.279 | 0.958 | 0.949 |
  | 4 oil_spot | 1.303 | 0.844–1.303 | 0.029 | 0.008 | 0.044 | 0.946 | 0.932 |

  Bộ phân lớp hạng 1 (chỉ hướng u): 91/241 ảnh đúng, 98/241 cùng nhãn với `full`.
- **Ý nghĩa**:
  - Dự đoán sai ở ba trên bốn điểm:
    - hướng riêng chính u gần như vuông góc với tâm lớp (|cos| 0.03–0.09);
    - năng lượng của u ở pc1–pc4 không cao hơn mốc ngẫu nhiên 0.0625 một cách nhất quán (0.008–0.159);
    - một hướng không đủ phân lớp (91/241, so với 205/241 của `q`).
  - Như vậy q không phải "bộ lọc theo hướng tâm lớp". Thông tin của q trải trên nhiều hướng.
  - Camera: u của water_spot có 0.279 năng lượng ở bốn thành phần mang camera (4.5 lần mốc). Ở các lớp khác là 0.04–0.15. Không có dấu hiệu chung rằng hướng chính của Σ là camera.
  - Dấu của phần ngoài đường chéo của E trùng dấu của dữ liệu ở 93–96% phần tử, với cả M lẫn S. Hai tỉ lệ gần bằng nhau (chênh 0.006–0.016), nên phép đo này không phân biệt được Adam đẩy theo M hay theo S. Điều nó cho thấy: Σ_c − I gần như là "ngân sách bước × dấu của hiệp phương sai lớp", đúng kiểu cập nhật theo dấu của Adam (D23).
  - [SUY LUẬN] Từ đó, q_c(x) ≈ ½ xᵀ E_c x ≈ ½ η Σ_jk sign(S_c)_jk x_j x_k. Tức q chấm xem cách các thành phần của x cùng tăng giảm có khớp với mẫu dấu tương quan của lớp c không, chủ yếu qua phần tử ngoài đường chéo. Bộ phân lớp là một dạng QDA với hiệp phương sai bị lượng tử hoá theo dấu, và độ lớn tỉ lệ với số bước.

## 4. Lỗi trên test S0

### 4.1 Ma trận nhầm, ảnh nhiều lỗi và camera (Q08.03)

Hỏi:
1. 35 ảnh sai (241 − 206) nằm ở cặp lớp nào?
2. Q08.03: trong từng lớp, độ chính xác trên ảnh chỉ có một loại lỗi khác bao nhiêu so với ảnh có nhiều loại lỗi? Trong các ảnh sai, bao nhiêu ảnh bị đoán thành một lớp có hộp lỗi ngay trong ảnh đó (nhãn mơ hồ, không hẳn là lỗi của mô hình)?
3. Ảnh sai có dồn theo camera không?

**Cách làm**:
- Gắn mỗi hàng của `test0_x` với ảnh của nó theo byte trong file `test_features` (như `05a-camera` §1.1). Đọc hộp lỗi bằng `data_io.objects`.
- "Nhiều lỗi": XML có hơn một tên lỗi, cùng quy tắc với `diagnostics` (`00-data-gc10` §3.2).
- Camera chia hai nhóm theo độ sáng (`00-data-gc10` §2.2): tối (04, 05, 08) và sáng (còn lại).
- In thêm hộp lỗi của ảnh test 0 (ví dụ ở 3.1).

Dự đoán [SUY LUẬN, từ `00-data-gc10` §3.4, §3.6]: phần lớn ảnh sai nằm giữa ba lớp quanh mối hàn (punching_hole, welding_line, crescent_gap); một phần lớn trong số đó là ảnh nhiều lỗi có hộp của lớp bị đoán.

In [7]:
# cell 6: S0 confusion matrix of the pipeline (full); test images mapped to their images by bytes, XML boxes (data_io.objects),
#         cameras; Q08.03: accuracy on single- vs multi-defect images per class, errors whose predicted class has a box in the image
import data_io
F_41 = Path(st_31["args"].data_dir) / st_31["args"].pretrained_model_name
Xt_41 = np.load(F_41 / f"test_features-{F_41.name}.npy")
pos_41 = {r.tobytes(): i for i, r in enumerate(Xt_41)}
img_41 = data_io.images("gc10")
te_41 = img_41[img_41["part"] == "test"].set_index("row").sort_index()
meta_41 = te_41.loc[[pos_41[r.tobytes()] for r in np.asarray(st_31["test0_x"])]].reset_index()
assert np.array_equal(meta_41["cls"].to_numpy(), y_31), "test rows mapped to the wrong images"
names_41 = [n for _, n in sorted(set(zip(img_41["cls"], img_41["name"])))]
objs_41 = [data_io.objects("gc10", p) or [] for p in meta_41["path"]]
meta_41["boxes"] = [sorted({o["cls"] for o in ob if o["cls"] is not None}) for ob in objs_41]
meta_41["multi"] = [len({o["name"] for o in ob}) > 1 for ob in objs_41]
meta_41["pred"] = full_32
meta_41["ok"] = meta_41["pred"] == meta_41["cls"]
meta_41["dark"] = meta_41["cam"].isin(["04", "05", "08"])

cm_41 = pd.crosstab(pd.Categorical(meta_41["name"], names_41[:5]), pd.Categorical([names_41[p] for p in meta_41["pred"]], names_41),
                    dropna=False, rownames=["true"], colnames=["pred"])
print(cm_41.loc[:, names_41[:5]].to_string(), "\npredicted into slots 5-9:", int((meta_41["pred"] >= 5).sum()))

by_41 = []
for c in range(5):
    g = meta_41[meta_41["cls"] == c]
    s, m = g[~g["multi"]], g[g["multi"]]
    by_41.append(dict(cls=names_41[c], n=len(g), multi=len(m), ok_single=f"{s['ok'].sum()}/{len(s)}", ok_multi=f"{m['ok'].sum()}/{len(m)}",
                      ok_bright=f"{g[~g['dark']]['ok'].sum()}/{(~g['dark']).sum()}", ok_dark=f"{g[g['dark']]['ok'].sum()}/{g['dark'].sum()}"))
print("\n" + pd.DataFrame(by_41).to_string(index=False))

err_41 = meta_41[~meta_41["ok"]]
box_hit_41 = [p in b for p, b in zip(err_41["pred"], err_41["boxes"])]
print(f"\nerrors {len(err_41)}: multi-defect {int(err_41['multi'].sum())}; predicted class has a box in the image {sum(box_hit_41)}")
pairs_41 = err_41.assign(box_hit=box_hit_41).groupby([err_41["name"], err_41["pred"].map(lambda p: names_41[p])])
print(pairs_41.agg(n=("ok", "size"), multi=("multi", "sum"), box_hit=("box_hit", "sum")).to_string())
print("\ntest image 0:", meta_41.loc[0, ["name", "cam", "coil", "boxes", "multi"]].to_dict(), "| predicted", names_41[meta_41.loc[0, "pred"]])

pred           punching_hole  welding_line  crescent_gap  water_spot  oil_spot
true                                                                          
punching_hole             39             4             0           0         0
welding_line               7            38             9           0         0
crescent_gap               1             2            42           0         0
water_spot                 0             0             0          50         8
oil_spot                   2             0             0           2        37 
predicted into slots 5-9: 0

          cls  n  multi ok_single ok_multi ok_bright ok_dark
punching_hole 43     31     10/12    29/31     39/43     0/0
 welding_line 54     19     28/35    10/19     29/39    9/15
 crescent_gap 45     24     21/21    21/24     14/16   28/29
   water_spot 58      2     48/56      2/2     50/58     0/0
     oil_spot 41      2     36/39      1/2     28/32     9/9

errors 35: multi-defect 15; predicted class has a 

- **Kết quả**:
  - Ma trận nhầm S0 (hàng là lớp thật, cột là lớp đoán):
    - đường chéo 39, 38, 42, 50, 37;
    - nhầm lớn nhất: welding_line → crescent_gap 9, water_spot → oil_spot 8, welding_line → punching_hole 7, punching_hole → welding_line 4;
    - không ảnh nào vào khe 5–9.
  - Theo lớp:

    | lớp | n | nhiều lỗi | đúng, một lỗi | đúng, nhiều lỗi | đúng, camera sáng | đúng, camera tối |
    |---|---|---|---|---|---|---|
    | punching_hole | 43 | 31 | 10/12 | 29/31 | 39/43 | 0/0 |
    | welding_line | 54 | 19 | 28/35 | 10/19 | 29/39 | 9/15 |
    | crescent_gap | 45 | 24 | 21/21 | 21/24 | 14/16 | 28/29 |
    | water_spot | 58 | 2 | 48/56 | 2/2 | 50/58 | 0/0 |
    | oil_spot | 41 | 2 | 36/39 | 1/2 | 28/32 | 9/9 |

  - 35 ảnh sai: 15 ảnh nhiều lỗi; 17 ảnh bị đoán thành một lớp có hộp lỗi ngay trong ảnh.
  - Theo cặp, số ảnh có hộp của lớp bị đoán:
    - welding_line → punching_hole: 7/7;
    - welding_line → crescent_gap: 3/9;
    - punching_hole → welding_line: 2/4;
    - crescent_gap → welding_line: 2/2;
    - crescent_gap → punching_hole: 1/1;
    - water_spot → oil_spot: 2/8;
    - các cặp còn lại: 0.
  - Ảnh test 0: punching_hole, camera 02, chỉ có hộp punching_hole, bị đoán là welding_line.
- **Ý nghĩa**:
  - Dự đoán đúng: 23/35 ảnh sai nằm trong ba lớp quanh mối hàn (tự cộng 4 + 9 + 7 + 1 + 2), và 15 trong 23 ảnh đó có hộp của lớp bị đoán (tự cộng).
  - Trả lời Q08.03: 17/35 ảnh sai (49%) bị đoán thành một lớp có mặt trong chính ảnh. Đây là nhãn mơ hồ (một ảnh, nhiều lỗi, một nhãn), không hẳn là lỗi của bộ phân lớp. Riêng welding_line → punching_hole thì cả 7 ảnh đều thế.
  - Có 2 ảnh trong số này chỉ có một loại lỗi (17 trùng hộp so với 15 nhiều lỗi): mỗi ảnh chỉ có hộp của lớp bị đoán, tức nhãn thư mục khác nhãn XML (`folder_not_in_xml`, `00-data-gc10` §3.1).
  - Ảnh nhiều lỗi chỉ làm hại welding_line (53% so với 80% ở ảnh một lỗi). punching_hole và crescent_gap đúng gần hết cả ảnh nhiều lỗi. [SUY LUẬN] Ảnh welding_line nhiều lỗi thường chứa một lỗi khác nằm trên mối hàn, và lỗi đó quyết định nhãn được đoán.
  - Camera không phải nguồn lỗi chính. Ở welding_line, camera tối kém hơn một chút (9/15 so với 29/39); ở crescent_gap ngược lại (28/29 so với 14/16). Ảnh test 0 là một lỗi thật của mô hình: chỉ có một lỗi, camera sáng.
  - [SUY LUẬN] Nếu tính các ảnh trùng hộp là đúng, độ chính xác S0 lên khoảng (206 + 17)/241 = 92.5%. Vậy S0_All và S1_Old có một trần khoảng 7 điểm do nhãn. QD25 nên báo thêm S1_Old trên tập test "sạch" (ảnh một lỗi, nhãn thư mục trùng XML) như một đại lượng phụ. Ghi vào Q08.03 khi đóng câu, chưa đổi QD25.

## 5. Qua 15 ảnh chụp

### 5.1 q quyết định, b vô hại, D24 và lỗi theo lớp có giữ qua seed không

Hỏi: các kết quả của 3.1, 3.2, 4.1 trên r0 s4 có đúng ở cả 15 ảnh chụp không?

**Cách làm**: đọc `snap/04`, `snap/05`, `snap/06.pkl` của 15 run. Tính lại các số hạng như 3.1 trên `z_test0` của chính ảnh chụp đó. In:
- số ảnh đúng. Phép kiểm: phải trùng S0_All của run (r0: 206, 206, 204, 205, 206; r1: 205, 205, 205, 203, 206; r2: 205, 208, 208, 207, 207);
- độ trải trung bình của b, q, ℓ giữa 5 lớp cũ;
- số ảnh mà |Δq| lớn nhất trong ba hiệu;
- số ảnh đổi nhãn khi bỏ b (đọc theo paper);
- số ảnh vào khe 5–9 (D24);
- recall từng lớp cũ.

Dự đoán [SUY LUẬN, từ `06-gauss-svi` §4.1: trong một FT seed các seed chỉ đổi tối đa 4/241 dự đoán]: ở cả 15 ảnh chụp, q quyết định, bỏ b đổi 0–2 ảnh, không ảnh nào vào khe 5–9, recall từng lớp gần như không đổi.

In [8]:
# cell 7: across the 15 snapshots (pickle): exact terms as in cell 3 on each snapshot's z_test0 with its params0; correct count
#         (check: S0_All of the runs), spreads, how often |dq| is the largest, labels changed by dropping b, slots 5-9 (D24), recall
import pickle

def terms_51(p, X):
    """b (C,), q (n, C), l (n, C), k (C,) of cell 3, float64."""
    mu, S = np.asarray(p["class_means"], np.float64), np.asarray(p["class_covs"], np.float64)
    P = np.linalg.inv(S)
    q = -0.5 * np.einsum("ni,cij,nj->nc", X, P - np.eye(X.shape[1]), X)
    return (-0.5 * np.linalg.slogdet(S)[1], q, np.einsum("ci,cij,nj->nc", mu, P, X), -0.5 * np.einsum("ci,cij,cj->c", mu, P, mu))

rows_51 = []
for d in sorted(Path("runs").glob("*__gc10_v2_ft_r[0-2]_s[0-4]")):
    tag = d.name.split("__")[1][len("gc10_v2_ft_"):]
    b4, b5, b6 = (pickle.loads((d / "snap" / f"{k}.pkl").read_bytes())["out"] for k in ("04", "05", "06"))
    X = np.asarray(b5["z_test0"], np.float64)
    y = np.asarray(b4["true_test0_y"]).astype(int)
    b, q, l, k = terms_51(b6["params0"], X)
    full, nob = b + q + l + k, q + l + k
    pf = full.argmax(1)
    top2 = np.argsort(-full, axis=1)[:, :2]
    ii = np.arange(len(y))
    dlt = np.stack([(T[ii, top2[:, 0]] - T[ii, top2[:, 1]]) for T in (np.broadcast_to(b, q.shape), q, l)], 1)
    sp = lambda T: (T[:, :5].max(1) - T[:, :5].min(1)).mean()
    rows_51.append(dict(snap=tag, correct=int((pf == y).sum()), spread_b=round(float(b[:5].max() - b[:5].min()), 3),
                        spread_q=round(float(sp(q)), 2), spread_l=round(float(sp(l)), 2),
                        q_largest=int((np.abs(dlt).argmax(1) == 1).sum()), no_b_changes=int((nob.argmax(1) != pf).sum()),
                        to_5_9=int((pf >= 5).sum()), **{f"rec_{c}": f"{((pf == c) & (y == c)).sum()}/{(y == c).sum()}" for c in range(5)}))
print(pd.DataFrame(rows_51).to_string(index=False))
print("classes 0-4:", names_41[:5])

 snap  correct  spread_b  spread_q  spread_l  q_largest  no_b_changes  to_5_9 rec_0 rec_1 rec_2 rec_3 rec_4
r0_s0      206     0.296     11.00      0.75        238             1       0 39/43 38/54 42/45 50/58 37/41
r0_s1      206     0.302     10.97      0.75        238             2       0 39/43 38/54 42/45 50/58 37/41
r0_s2      204     0.294     10.98      0.74        239             1       0 38/43 37/54 42/45 50/58 37/41
r0_s3      205     0.301     10.98      0.74        237             3       0 39/43 37/54 42/45 50/58 37/41
r0_s4      206     0.294     11.00      0.75        237             0       0 39/43 38/54 42/45 50/58 37/41
r1_s0      205     0.311     10.18      0.73        236             0       0 39/43 39/54 41/45 51/58 35/41
r1_s1      205     0.311     10.20      0.73        235             2       0 39/43 39/54 42/45 51/58 34/41
r1_s2      205     0.308     10.23      0.73        237             1       0 39/43 39/54 41/45 51/58 35/41
r1_s3      203     0.317    

- **Kết quả**:
  - Phép kiểm đạt: số ảnh đúng trùng S0_All của cả 15 run (r0: 206, 206, 204, 205, 206; r1: 205, 205, 205, 203, 206; r2: 205, 208, 208, 207, 207).
  - Độ trải trung bình giữa 5 lớp cũ: q 10.16–11.00, ℓ 0.73–0.75, b 0.294–0.341.
  - |Δq| lớn nhất trong ba hiệu ở 229–239/241 ảnh.
  - Bỏ b (đọc theo paper) đổi nhãn 0–3 ảnh.
  - Không ảnh nào vào khe 5–9, ở cả 15 ảnh chụp.
  - Recall từng lớp gần như cố định trong một FT seed: punching_hole 38–40/43, crescent_gap 41–42/45. Khác nhau chủ yếu giữa các FT seed:
    - welding_line: 37–38 (r0), 39 (r1), 35–36 (r2) trên 54;
    - water_spot: 50, 51, 52 trên 58;
    - oil_spot: 37 (r0), 33–35 (r1), 37–39 (r2) trên 41.
- **Ý nghĩa**:
  - Dự đoán đúng cả bốn ý: ở mọi ảnh chụp, q quyết định (q trải gấp khoảng 14 lần ℓ); log det gần như không có tác dụng ở S0; D24 không xảy ra với SVI trên GC10 ở S0; lỗi theo lớp ổn định qua seed.
  - Khác biệt giữa các seed (`06-gauss-svi` §4.1) và giữa các FT seed đều nằm trong recall của vài lớp, không đổi cơ chế. Đơn vị FT seed (n = 3) đổi độ chính xác S0 nhiều nhất ở welding_line và oil_spot.
  - D24 cho GC10, SVI, ở S0: đóng được cho 15/15 ảnh chụp. Ở S1, mọi khe đều đã học (Q08.01, ghi chú 2026-10-20), nên D24 không còn chỗ tác động.

## Summary (EN)

1. **Step 08 reproduces exactly and is a quadratic discriminant with equal priors.** `rerun` gives both keys identical to the snapshot; all four S0 rows score 206/241 (§1.1). `_predict` takes the argmax over all 10 slots of the full Gaussian log-density, log det included, with no class prior (§2.1).
2. **The decision is made by the quadratic term of Sigma − I, not by the means.** The exact split log N = b + q + l + k + common (b = −½ log det Σ, q = −½ xᵀ(Σ⁻¹ − I)x, l = μᵀΣ⁻¹x) reproduces the pipeline on 241/241 images. Across the old classes q spreads by 11.0 per image, l by 0.75, b by 0.29. q alone agrees with the pipeline on 240/241 images and is the largest term of the winner-vs-runner-up margin on 237/241 (§3.1–3.2). The means alone still reach 200/241, but they are about 0.09 long while ‖x‖² is about 190, so q dominates whenever the two disagree. This corrects the [inference] of `06-gauss-svi` §1.1.
3. **Sigma − I is the step budget times the sign of the class covariance.** Off-diagonal signs of Σ − I match the class data at 93–96% of entries. The leading eigen-direction is nearly orthogonal to the class centre (|cos| 0.03–0.09), and a rank-1 version of q gets only 91/241, so q spreads over many directions (§3.3). [inference] The classifier is a sign-quantised QDA whose strength grows with the number of Adam steps, which matters for the old-versus-new competition at steps 12 and 14.
4. **Paper vs code (candidate D25), and D24, are harmless at S0.** Dropping the log det term, as Eq. 4 and 6 of the paper read, changes 0–3 labels over the 15 snapshots, and no image ever goes to an untrained slot (§3.2, §5.1). Eq. 6 (a similarity exp(−½m²)) and Eq. 8 (a distance) also disagree within the paper.
5. **Half of the S0 errors are label ambiguity (Q08.03).** 23 of the 35 errors lie among the three weld classes. 17/35 errors predict a class that has a box in the same image, including all 7 welding_line → punching_hole errors. Multi-defect images hurt only welding_line (10/19 vs 28/35 correct). Camera is not a main error source (§4.1). Per-class recall is stable within an FT seed (§5.1). [inference] Counting box matches as correct would put S0 near 92.5%, a label ceiling of about 7 points on S0_All and S1_Old.